# 13.2 语音语言 (Audio-Language)

语音语言模型将语音信号与文本理解结合，实现语音识别、语音翻译、语音合成等任务。从Whisper到GPT-4o的语音模式，语音语言模型正在走向统一。

## 语音语言模型架构演进

| 阶段 | 代表模型 | 架构 | 能力 |
|------|----------|------|------|
| 级联式 | ASR+LLM+TTS | 三阶段流水线 | 语音对话 |
| 语音理解 | Whisper/Qwen-Audio | 音频编码+LLM | 语音→文本 |
| 语音生成 | VITS/Bark | 文本→语音 | 文本→语音 |
| 统一模型 | GPT-4o Audio | 单一Transformer | 语音↔文本 |

## 1. 语音编码器

语音编码器将音频信号转换为特征表示，是语音语言模型的基础。

### 音频预处理
- **梅尔频谱图**：将音频转换为时频表示，最常用的音频特征
- **MFCC**：梅尔频率倒谱系数，传统语音识别特征
- **EnCodec**：Meta的神经音频编解码器，将音频压缩为离散token

### Whisper编码器
- 输入：80维梅尔频谱图
- 架构：Conv下采样 + Transformer编码器
- 特点：680K小时多语言训练，极强的鲁棒性

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class MelSpectrogramSimulator:
    def __init__(self, n_mels=80, n_fft=400, hop_length=160, sample_rate=16000):
        self.n_mels = n_mels
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.sample_rate = sample_rate

    def forward(self, waveform):
        B, T = waveform.shape
        n_frames = T // self.hop_length
        mel = torch.randn(B, self.n_mels, n_frames) * 0.5
        mel = F.layer_norm(mel, mel.shape[1:])
        return mel

class WhisperStyleEncoder(nn.Module):
    def __init__(self, n_mels=80, d_model=128, n_heads=2, n_layers=2):
        super().__init__()
        self.conv1 = nn.Conv1d(n_mels, d_model, kernel_size=3, padding=1)
        self.conv2 = nn.Conv1d(d_model, d_model, kernel_size=3, stride=2, padding=1)
        self.pos_embed = nn.Parameter(torch.randn(1, 1000, d_model) * 0.02)
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4, batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, n_layers)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, mel_spec):
        x = F.gelu(self.conv1(mel_spec))
        x = F.gelu(self.conv2(x))
        x = x.transpose(1, 2)
        T = x.shape[1]
        x = x + self.pos_embed[:, :T, :]
        x = self.encoder(x)
        return self.norm(x)

class EnCodecStyleTokenizer(nn.Module):
    def __init__(self, d_model=128, n_codebooks=4, codebook_size=1024):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv1d(1, d_model, 10, 5, 3),
            nn.GELU(),
            nn.Conv1d(d_model, d_model, 3, 2, 1),
            nn.GELU(),
        )
        self.quantizer = nn.ModuleList([
            nn.Linear(d_model, codebook_size) for _ in range(n_codebooks)
        ])
        self.n_codebooks = n_codebooks

    def forward(self, waveform):
        x = waveform.unsqueeze(1)
        h = self.encoder(x)
        h = h.transpose(1, 2)
        codes = []
        for i, cb in enumerate(self.quantizer):
            code = cb(h).argmax(dim=-1)
            codes.append(code)
        return torch.stack(codes, dim=1)

mel_sim = MelSpectrogramSimulator(n_mels=80)
waveform = torch.randn(2, 16000)
mel = mel_sim.forward(waveform)

whisper_enc = WhisperStyleEncoder(n_mels=80, d_model=128)
audio_features = whisper_enc(mel)

encodec = EnCodecStyleTokenizer(d_model=128, n_codebooks=4, codebook_size=1024)
audio_codes = encodec(waveform)

print('=== Audio Encoders ===')
print(f'Waveform: {waveform.shape}')
print(f'Mel spectrogram: {mel.shape}')
print(f'Whisper features: {audio_features.shape}')
print(f'\nEnCodec tokens: {audio_codes.shape} (n_codebooks x seq_len)')
print(f'  Codebook 0: {audio_codes[0, 0, :10].tolist()}...')
print(f'  Codebook 1: {audio_codes[0, 1, :10].tolist()}...')

print(f'\nKey: Whisper outputs continuous features for understanding.')
print(f'EnCodec outputs discrete tokens for generation.')

## 2. 语音语言模型

语音语言模型将音频特征与LLM结合，支持语音输入理解和语音输出。

### 架构选择
- **音频编码+LLM**：Whisper编码音频 → 投影到语言空间 → LLM生成文本
- **音频token+LLM**：EnCodec离散化音频 → 音频token与文本token一起输入LLM
- **统一模型**：音频和文本在同一个Transformer中处理

### 关键挑战
- **音频序列长度**：音频比文本长10-100倍，需要下采样
- **模态对齐**：音频和文本的语义粒度不同
- **实时性**：语音对话需要低延迟

In [ ]:
class AudioLanguageModel(nn.Module):
    def __init__(self, d_audio=128, d_text=128, vocab_size=500, n_audio_tokens=50):
        super().__init__()
        self.audio_encoder = WhisperStyleEncoder(n_mels=80, d_model=d_audio)
        self.audio_downsample = nn.Conv1d(d_audio, d_text, kernel_size=5, stride=4, padding=2)
        self.audio_proj = nn.Linear(d_text, d_text)
        self.text_embed = nn.Embedding(vocab_size, d_text)
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_text, 2, d_text * 4, batch_first=True), 2
        )
        self.head = nn.Linear(d_text, vocab_size)

    def forward(self, mel_spec, text_tokens):
        audio_features = self.audio_encoder(mel_spec)
        audio_features = audio_features.transpose(1, 2)
        audio_down = self.audio_downsample(audio_features).transpose(1, 2)
        audio_tokens = self.audio_proj(audio_down)
        txt_tokens = self.text_embed(text_tokens)
        combined = torch.cat([audio_tokens, txt_tokens], dim=1)
        h = self.lm(combined)
        text_output = h[:, audio_tokens.shape[1]:]
        return self.head(text_output)

class UnifiedAudioTextModel(nn.Module):
    def __init__(self, d_model=128, vocab_size=500, audio_vocab_size=1024, n_codebooks=4):
        super().__init__()
        self.text_embed = nn.Embedding(vocab_size, d_model)
        self.audio_embed = nn.ModuleList([
            nn.Embedding(audio_vocab_size, d_model) for _ in range(n_codebooks)
        ])
        self.n_codebooks = n_codebooks
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 2, d_model * 4, batch_first=True), 2
        )
        self.text_head = nn.Linear(d_model, vocab_size)
        self.audio_heads = nn.ModuleList([
            nn.Linear(d_model, audio_vocab_size) for _ in range(n_codebooks)
        ])

    def forward(self, tokens, token_types):
        embeddings = torch.zeros(tokens.shape[0], tokens.shape[1],
                                  self.text_embed.embedding_dim, device=tokens.device)
        text_mask = token_types == 0
        audio_mask = token_types == 1
        embeddings[text_mask] = self.text_embed(tokens[text_mask])
        for cb_idx in range(self.n_codebooks):
            cb_mask = token_types == (cb_idx + 2)
            if cb_mask.any():
                embeddings[cb_mask] = self.audio_embed[cb_idx](tokens[cb_mask])
        h = self.lm(embeddings)
        return h

mel_sim = MelSpectrogramSimulator(n_mels=80)
waveform = torch.randn(2, 16000)
mel = mel_sim.forward(waveform)
text = torch.randint(0, 500, (2, 10))

alm = AudioLanguageModel()
logits = alm(mel, text)

print('=== Audio-Language Model ===')
print(f'Mel spectrogram: {mel.shape}')
print(f'Text tokens: {text.shape}')
print(f'Output logits: {logits.shape}')

unified = UnifiedAudioTextModel()
tokens = torch.randint(0, 500, (2, 20))
token_types = torch.randint(0, 6, (2, 20))
h = unified(tokens, token_types)

print(f'\nUnified model hidden: {h.shape}')
print(f'\nKey: Audio downsample reduces sequence length for efficient LLM processing.')
print(f'Unified model handles both audio and text tokens natively.')

## 3. 语音合成 (TTS)

现代TTS系统从传统流水线发展为端到端和基于语言模型的方法。

### TTS方法演进
| 方法 | 架构 | 优点 | 缺点 |
|------|------|------|------|
| VITS | 端到端VAE | 高质量、快速 | 训练不稳定 |
| Bark | 自回归LM | 多语言、多风格 | 速度慢 |
| XTTS | 自回归LM+编解码器 | 语音克隆 | 需要参考音频 |
| Voicebox | 流匹配 | 快速、可编辑 | 训练复杂 |
| SpeechX | 统一模型 | ASR+TTS一体 | 计算量大 |

### LLM-based TTS核心流程
1. 文本 → 文本token
2. LLM自回归生成音频token
3. 音频token → EnCodec解码器 → 波形

In [ ]:
class LLMBasedTTS(nn.Module):
    def __init__(self, d_model=128, text_vocab=500, audio_vocab=1024, n_codebooks=4):
        super().__init__()
        self.text_embed = nn.Embedding(text_vocab, d_model)
        self.audio_embed = nn.ModuleList([
            nn.Embedding(audio_vocab, d_model) for _ in range(n_codebooks)
        ])
        self.n_codebooks = n_codebooks
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, 2, d_model * 4, batch_first=True), 2
        )
        self.audio_heads = nn.ModuleList([
            nn.Linear(d_model, audio_vocab) for _ in range(n_codebooks)
        ])
        self.delay_pattern = [0, 1, 2, 3]

    def forward(self, text_tokens, audio_tokens=None):
        txt_emb = self.text_embed(text_tokens)
        if audio_tokens is not None:
            audio_emb = sum(self.audio_embed[i](audio_tokens[:, i])
                           for i in range(self.n_codebooks))
            combined = torch.cat([txt_emb, audio_emb], dim=1)
        else:
            combined = txt_emb
        h = self.lm(combined)
        audio_logits = [head(h) for head in self.audio_heads]
        return audio_logits

    @torch.no_grad()
    def generate(self, text_tokens, max_audio_len=50, temperature=1.0):
        txt_emb = self.text_embed(text_tokens)
        generated_codes = [[] for _ in range(self.n_codebooks)]
        current_input = txt_emb

        for step in range(max_audio_len):
            h = self.lm(current_input)
            last_h = h[:, -1:, :]
            for cb_idx in range(self.n_codebooks):
                logits = self.audio_heads[cb_idx](last_h) / temperature
                code = logits.argmax(dim=-1)
                generated_codes[cb_idx].append(code)

            new_emb = sum(self.audio_embed[i](generated_codes[i][-1])
                         for i in range(self.n_codebooks))
            current_input = torch.cat([current_input, new_emb], dim=1)

        return torch.stack([torch.cat(codes, dim=1) for codes in generated_codes], dim=1)

tts = LLMBasedTTS()
text_tokens = torch.randint(0, 500, (1, 8))
audio_codes = tts.generate(text_tokens, max_audio_len=20, temperature=0.8)

print('=== LLM-based TTS ===')
print(f'Text tokens: {text_tokens.shape}')
print(f'Generated audio codes: {audio_codes.shape}')
print(f'  Codebook 0: {audio_codes[0, 0, :10].tolist()}...')
print(f'  Codebook 1: {audio_codes[0, 1, :10].tolist()}...')
print(f'  Codebook 2: {audio_codes[0, 2, :10].tolist()}...')
print(f'  Codebook 3: {audio_codes[0, 3, :10].tolist()}...')

print(f'\nKey: LLM-based TTS generates audio tokens autoregressively.')
print(f'Multiple codebooks capture different levels of audio detail.')
print(f'Delay pattern allows parallel decoding across codebooks.')

## 4. 语音语言模型工业实践

### 级联式 vs 统一式
| 维度 | 级联式(ASR+LLM+TTS) | 统一式(GPT-4o Audio) |
|------|----------------------|----------------------|
| 延迟 | 高（3阶段串行） | 低（端到端） |
| 情感 | 丢失（文本中间表示） | 保留（音频直接传递） |
| 质量 | 各阶段独立优化 | 端到端优化 |
| 复杂度 | 低（组合现有模型） | 高（需要大量数据训练） |

### 最佳实践
1. **快速原型**：级联式（Whisper + LLM + TTS），最快上线
2. **追求质量**：统一式，需要大量训练资源
3. **音频下采样**：4-8倍下采样是必要的，否则序列太长
4. **多码本策略**：EnCodec 4个码本足够，8个码本质量更好
5. **语音克隆**：需要3-10秒参考音频，用提示学习或LoRA实现